# project_17_nanobody_taa — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — TAA biology, CDR structure, epitope choice + the VHH hello-world

**Standard slot:** *define & explore.* **For Project 17 this means:** understand the tumor-associated
antigen (TAA) and the nanobody (VHH) you will design against it, **choose your epitope** (overlapping
vs non-overlapping with an approved mAb), fix the metrics table, and run the **mock** VHH hello-world
end-to-end (D0).

Run `00_setup.ipynb` first in this session. Everything here runs with **no GPU** on the deterministic
`mock` backend — switch to the real backend (RFantibody/BoltzGen) on Colab A100 in notebook 02.

## The biology in one screen

A **nanobody (VHH)** is the ~15 kDa single variable domain of a camelid heavy-chain-only antibody. It
folds as one immunoglobulin domain with three **CDR loops** (CDR1, CDR2, and the long, dominant
**CDR3**) presented on a stable framework. That small, stable, single-domain format is why nanobodies
power **tumor-imaging agents** (fast clearance, deep penetration) and **CAR / bispecific** binder
modules.

A **tumor-associated antigen (TAA)** is a cell-surface protein over-expressed on tumor cells. Classic
examples: **HER2** (ERBB2) and **EGFR** (ERBB1) in the HER/ErbB receptor family, and **mesothelin**.
De novo VHH design to a *defined, validated* TAA epitope is a real translational pipeline — but it is
hard, and hit rates are LOW, so we treat designs as **screening inputs**, not finished binders.

**The epitope choice is the central design decision (P0/P1).** Two strategies:
- **Overlapping** with an approved mAb's footprint (e.g., the trastuzumab epitope on HER2 domain IV)
  → you compete with / mimic a validated therapeutic site.
- **Non-overlapping / orthogonal** → enables **biparatopic** or **bispecific** constructs that bind
  alongside the mAb, and avoids resistance tied to the mAb site.

You will justify your choice with explicit, measurable success criteria in your D0 problem statement.

## The metrics table (what we will filter on)

| Metric | Range | Means | Does **not** mean | Cutoff (antibody) |
|--------|-------|-------|-------------------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the VHH model | thermostability / affinity | ≥ 70 |
| pae_interaction | Å | confidence in the VHH–antigen *interface* arrangement | binding/affinity | ≤ 12 |
| scRMSD | Å | designed-vs-predicted VHH backbone (self-consistency) | binding | ≤ 3.0 |
| CDR geometry RMSD | Å | CDR-loop geometry vs an IgFold/NanoBodyBuilder2 model | a good paratope | small (loop sanity) |
| TAP-like score | count | developability liability flags (proxy) | a real TAP call | fewer = better |
| CamSol-like | a.u. | solubility proxy (hydropathy) | a real CamSol score | higher = better |
| humanness | 0–1 | human-likeness proxy | low immunogenicity guarantee | higher = better |

The `"antibody"` cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12) come from the shared
`filtering_pipeline.DEFAULT_CUTOFFS["antibody"]`. **Developability metrics here are TEACHING
HEURISTICS, not the validated tools (TAP/CamSol/Hu-mAb)** — see `MANUAL.md §2` and `antibody_tools.py`.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Pick your TAA, epitope, and framework

These are the three fixed inputs to the campaign. The structures are **candidates — verify on RCSB in
Week 1** (`data/README.md`). The epitope residue lists below are **EXAMPLE placeholders** — derive the
real ones from the antigen surface (and, for the overlapping choice, from the mAb–antigen interface in
the reference complex).

In [ ]:
# --- Campaign definition (EDIT these in Week 1 after verifying the structures) ---
TAA = "HER2"                 # one of: HER2 (1N8Z), EGFR (1IVO), mesothelin (model) — verify on RCSB
REFERENCE_COMPLEX = "1N8Z"   # candidate: trastuzumab Fab - HER2 domain IV (verify; entries get superseded)

# EXAMPLE epitope residue tokens (chain+number). REPLACE with residues you read off the antigen
# surface. For the "overlapping" strategy, take residues from the mAb-antigen interface in 1N8Z;
# for "non-overlapping", choose a distinct patch (e.g., HER2 domain I/II away from the trastuzumab site).
EPITOPE_OVERLAPPING     = "A557,A560,A579,A580,A583"   # EXAMPLE — trastuzumab-region (verify!)
EPITOPE_NONOVERLAPPING  = "A245,A266,A270,A289"        # EXAMPLE — orthogonal patch (verify!)

# Choose one to drive the campaign; record WHY in your D0 problem statement.
EPITOPE = EPITOPE_OVERLAPPING
EPITOPE_STRATEGY = "overlapping-with-trastuzumab"   # or "non-overlapping-orthogonal"

print("TAA            =", TAA)
print("epitope        =", EPITOPE, "(", EPITOPE_STRATEGY, ")")
print("ref complex    =", REFERENCE_COMPLEX, "(candidate — verify on RCSB)")

## The VHH framework

The CDRs are the design variables; the **framework is fixed**. `antibody_tools.DEFAULT_FRAMEWORK` is a
humanized-VHH (huVHH3-style) **teaching placeholder** — verify and replace with the exact germline
framework you choose. Keeping a humanized framework helps the humanness axis from the start.

In [ ]:
from antibody_tools import DEFAULT_FRAMEWORK, parse_epitope, epitope_overlap

fw = DEFAULT_FRAMEWORK
print("framework:", fw["name"])
for k in ("FR1", "FR2", "FR3", "FR4"):
    print(f"  {k}: {fw[k]}")

ep = parse_epitope(EPITOPE)
print("\nparsed epitope residues:", ep)

## VHH hello-world (mock backend, no GPU)

Generate a few mock VHH designs against your epitope and assemble + score one. This proves the
plumbing (design → CDR loops → AF2-Multimer-ab metrics → developability) before you spend A100 time in
notebook 02. **Every number below is SYNTHETIC — never report mock numbers as real.**

In [ ]:
from antibody_tools import design_vhh_cdrs, score_designs, extract_cdrs

# Mock hello-world: 5 deterministic VHH candidates.
designs = design_vhh_cdrs(TAA, EPITOPE, framework=fw, n=5, tool="mock")
score_designs(designs, tool="mock")     # fills af2_multimer_ab + developability (SYNTHETIC)

d = designs[0]
print("design_id :", d.design_id)
print("VHH length:", len(d.sequence), "aa")
print("CDRs      :", extract_cdrs(d))
print("metrics   : pLDDT", d.plddt, "| pae_interaction", d.pae_interaction,
      "| scrmsd", d.scrmsd, "| cdr_geom", d.cdr_geom)
print("dev (heur):", "TAP-like", d.tap_score, "| CamSol-like", d.camsol_like,
      "| humanness", d.humanness)
print("epitope overlap:", epitope_overlap(d.contact_residues, EPITOPE))
print("synthetic :", d.synthetic, "->", d.notes)

## Visualize a structure (py3Dmol)

Use this to eyeball a real predicted VHH–antigen complex once you have one (notebook 02 on Colab). The
mock backend writes no PDB.

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real RFantibody/AF2-Multimer run):
# show_pdb("results/af2/vhh_her2_complex.pdb")
print("show_pdb(pdb_path) ready — use it on a real VHH-antigen complex in notebook 02.")

## D0 checklist
- [ ] 1-page **problem statement**: the TAA, the chosen **epitope + strategy** (overlapping vs
      non-overlapping, with justification), and **measurable** success criteria.
- [ ] Verified `REFERENCE_COMPLEX` / TAA accessions on RCSB; epitope residues read off the real
      surface (not the EXAMPLE placeholders).
- [ ] Metric table understood, including the "does not mean" column and that developability numbers
      here are **heuristics**.
- [ ] Mock VHH hello-world run; CDR3 length + SYNTHETIC metrics printed.
- [ ] `LOG.md` entry (seed, what you ran).

**Next:** `02_generate.ipynb` — the VHH design campaign (mock now; RFantibody on A100).

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — the VHH design campaign → results CSV

**Standard slot:** *design campaign.* **For Project 17 this means:** run the de novo VHH campaign —
RFantibody (RFdiffusion-Ab diffuses CDR loops onto the framework against your epitope, then ProteinMPNN
designs the loop sequence), or BoltzGen nanobody mode — score each (VHH, antigen) complex, and write a
results CSV (D2).

**Compute reality (be honest):** the real campaign wants an **A100**. A free T4 runs only a **tiny
RFantibody demo**. De novo nanobody hit rates are **LOW**, so you generate **many** (500+ where
feasible) and send survivors to a **display screen** (notebook 05) — designs are *screening inputs*,
not finished binders. This notebook runs on the **mock** backend so the plumbing executes anywhere.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (rule 4)

Tools change. Before a real run, confirm the pinned upstream repos still exist and **pin the exact
commit** you used (put it in `LOG.md`). This HTTP-checks the URLs; it does not install anything.

In [ ]:
import requests

# Pinned upstreams for the antibody family (pin the COMMIT you actually use — these move).
UPSTREAMS = {
    "RFantibody (RFdiffusion-Ab + ProteinMPNN)": "https://github.com/RosettaCommons/RFantibody",
    "ImmuneBuilder / NanoBodyBuilder2 (IgFold-style)": "https://github.com/oxpig/ImmuneBuilder",
    "ColabFold (AF2-Multimer)": "https://github.com/sokrypton/ColabFold",
    # BoltzGen: VERIFY the current public release at course start and pin it here (URL changes).
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"[{r.status_code}] {name}\n        {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e}\n        {url}")
print("\nNOTE: BoltzGen — verify the current public release/repo manually and pin it (MANUAL.md §2).")
print("Pin the exact COMMIT/tag of each tool in LOG.md before any real campaign.")

## Campaign parameters

Re-state the fixed inputs (same as notebook 01) and the campaign scale. On a real A100 run set
`N_DESIGNS` to **500+** (low hit rate ⇒ generate a large pool ⇒ filter hard ⇒ screen survivors). The
mock run uses a small N so it is fast everywhere.

In [ ]:
from antibody_tools import DEFAULT_FRAMEWORK

TAA = "HER2"
EPITOPE = "A557,A560,A579,A580,A583"     # EXAMPLE — use your verified residues from notebook 01
FRAMEWORK = DEFAULT_FRAMEWORK

# Scale: mock=small so the notebook is fast; real A100 campaign -> N_DESIGNS=500+.
N_DESIGNS = 40           # -> 500+ on A100
TOOL = "mock"            # -> "rfantibody" (A100) or "boltzgen" (verify release) on Colab

print(f"campaign: {N_DESIGNS} VHH vs {TAA} @ {EPITOPE}  (tool={TOOL})")
print("Diversity BEFORE filtering: generate many CDR variants, filter aggressively in nb 03.")

## Run the campaign (mock) → score → CSV

`design_vhh_cdrs()` generates VHH candidates (CDR1/CDR2/CDR3 on the fixed framework);
`score_designs()` fills the AF2-Multimer-ab metrics **and** the developability heuristics. Switch
`TOOL` to `"rfantibody"` / `"boltzgen"` on an A100 to run for real (the functions raise a clear,
actionable `NotImplementedError` with the TODO until then).

In [ ]:
import pandas as pd
from antibody_tools import design_vhh_cdrs, score_designs

designs = design_vhh_cdrs(TAA, EPITOPE, framework=FRAMEWORK, n=N_DESIGNS, tool=TOOL)
score_designs(designs, tool=TOOL)

rows = [d.as_row() for d in designs]
camp = pd.DataFrame(rows)
# Keep the columns the filter + analysis need; drop the bulky notes/list fields for the CSV view.
cols = ["design_id", "tool", "antigen", "framework", "cdr1", "cdr2", "cdr3",
        "plddt", "pae_interaction", "scrmsd", "cdr_geom",
        "tap_score", "camsol_like", "humanness", "synthetic"]
camp = camp[[c for c in cols if c in camp.columns]]
camp["cdr3_len"] = camp["cdr3"].str.len()
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape)
print("SYNTHETIC?" , bool(camp["synthetic"].all()), "(mock => all numbers are EXAMPLE_DATA)")
camp.head()

## Quick campaign sanity look

Before filtering, eyeball the distributions: CDR3 length, the key interface metric (pae_interaction),
and the developability heuristics. On the **mock** backend these are SYNTHETIC and only show the
plumbing; on a real run they tell you whether the pool is diverse and worth filtering.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].hist(camp["cdr3_len"], bins=12); ax[0].set_title("CDR3 length"); ax[0].set_xlabel("aa")
ax[1].hist(camp["pae_interaction"], bins=12); ax[1].set_title("pae_interaction (mock)"); ax[1].set_xlabel("Å")
ax[2].hist(camp["humanness"], bins=12); ax[2].set_title("humanness (heuristic)"); ax[2].set_xlabel("0–1")
plt.suptitle("Campaign pool — SYNTHETIC (mock) distributions; for plumbing only")
plt.tight_layout(); plt.savefig("results/campaign_distributions.png", dpi=150); plt.show()
print("Reminder: mock distributions are SYNTHETIC — real shape comes from the A100 campaign.")

## D2 checklist
- [ ] `results/campaign.csv`: the VHH pool (CDRs + metrics), one row per design.
- [ ] Version-verify cell run; exact tool **commits** pinned in `LOG.md`.
- [ ] Design log: framework, epitope, N, seed, tool/version, runtime per design.
- [ ] (Real run) campaign at **500+** on A100; note the realistic LOW hit rate and that survivors go
      to a display screen, not straight to "binder".
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — the shared antibody filter.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (`design_type="antibody"`)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 17** you map your VHH campaign onto `fp.Design` objects, run the pipeline with the
**antibody** cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12), and report survival (D3 pt 1).

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"antibody"` cutoffs.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)
print("\nUsing design_type='antibody':", fp.DEFAULT_CUTOFFS["antibody"])

## Build `fp.Design` objects from the campaign

The filter operates on `fp.Design` records. Map each VHH's AF2-Multimer-ab metrics onto the matching
fields. For an antibody complex the key fields are `plddt`, `pae_interaction`, and `scrmsd`; we stash
the developability heuristics + CDR3 length in `extra` so they ride along into the ranked CSV.

In [ ]:
camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    designs.append(fp.Design(
        design_id=str(r["design_id"]),
        sequence="",                      # full VHH not needed for the confidence layers
        design_type="antibody",
        plddt=r.get("plddt"),
        pae_interaction=r.get("pae_interaction"),
        scrmsd=r.get("scrmsd"),
        # solubility maps to the CamSol-like heuristic so Layer 3 (physics) has something to act on:
        solubility=r.get("camsol_like"),
        extra={"tap_score": r.get("tap_score"), "humanness": r.get("humanness"),
               "cdr_geom": r.get("cdr_geom"), "cdr3_len": r.get("cdr3_len"),
               "synthetic": bool(r.get("synthetic", False))},
    ))
print(len(designs), "fp.Design objects built (design_type='antibody')")

## Run the pipeline + report

`run_pipeline(..., design_type="antibody")` applies the layers in order with the antibody cutoffs and
returns a ranked DataFrame; `report()` prints the **survival-at-each-layer** accounting and saves the
ranked CSV + figure. We run Layers 1 + 3 here (self-consistency + physics/solubility); Layer 2
(orthogonal predictor agreement) needs a second predictor — wire IgFold/ESMFold scRMSD in for the real
run. **On mock data the survivors are SYNTHETIC** — the point is the plumbing and the honest accounting.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="antibody", use_layers=(1, 3))
top = fp.report(df_ranked, top_n=15, save_prefix="results/proj17")
print("\nranked CSV -> results/proj17_ranked.csv ; survival figure -> results/proj17_survival.png")
top

## Hit-rate accounting (report the rate, not the cherry)

Survival = how many of the generated pool pass each layer. For de novo nanobodies this is **low by
design** — that is the honest, expected result, and it is exactly why survivors go to a display screen
rather than straight to characterization.

In [ ]:
import pandas as pd
n_total = df_ranked.attrs.get("n_total", len(df_ranked))
survival = df_ranked.attrs.get("survival", {})
print(f"Generated: {n_total}")
for layer, n in survival.items():
    print(f"  {layer}: {n} survivors ({100*n/max(n_total,1):.1f}%)")
print("\nlayers_passed distribution:")
if "layers_passed" in df_ranked:
    print(df_ranked["layers_passed"].value_counts().sort_index())
print("\nReminder: mock survivors are SYNTHETIC. On a real campaign, expect a LOW pass rate — "
      "frame survivors as display-screen inputs, not finished binders.")

## D3 (part 1) checklist
- [ ] `results/proj17_ranked.csv` produced by the **shared** module with `design_type="antibody"`.
- [ ] Survival-at-each-layer reported (the survival figure saved).
- [ ] Mapping assumptions written down (which metric → which `fp.Design` field).
- [ ] Honest hit-rate accounting; survivors framed as screening inputs.

**Next:** `04_validate.ipynb` — epitope-choice + receptor-family specificity + developability figures.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — epitope choice, receptor-family specificity, developability

**Standard slot:** *validate (in silico).* **For Project 17 the core comparisons are:** (1) the
**epitope choice** (overlapping vs non-overlapping with an approved mAb), (2) **specificity within the
receptor family** (does the VHH prefer HER2 over EGFR/HER3/HER4?), and (3) **developability/humanness**
distributions of the survivors (D3 pt 2).

Needs `results/campaign.csv`. All metrics on the mock backend are **SYNTHETIC** — the figures here
demonstrate the analysis; real numbers come from the A100 campaign.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Epitope choice — overlapping vs non-overlapping with an approved mAb `[core]`

Re-run the campaign for **both** epitope strategies and compare the pools. The question is not "which
binds better" (mock can't answer that) but **what each strategy buys you**: an overlapping epitope
competes with / mimics a validated therapeutic site; a non-overlapping one enables biparatopic /
bispecific constructs and dodges resistance tied to the mAb site. Use `epitope_overlap()` against the
approved-mAb footprint to bin each design as competing vs orthogonal.

In [ ]:
import pandas as pd
from antibody_tools import design_vhh_cdrs, score_designs, epitope_overlap, DEFAULT_FRAMEWORK

TAA = "HER2"
FRAMEWORK = DEFAULT_FRAMEWORK
# Approved-mAb footprint (EXAMPLE — read trastuzumab's HER2 contacts off 1N8Z; verify!)
MAB_FOOTPRINT = "A557,A560,A579,A580,A583"
EPITOPES = {
    "overlapping":     "A557,A560,A579,A580,A583",   # same region as the mAb -> competes
    "non_overlapping": "A245,A266,A270,A289",         # distinct patch -> orthogonal / biparatopic
}

summary = []
for strat, ep in EPITOPES.items():
    ds = design_vhh_cdrs(TAA, ep, framework=FRAMEWORK, n=40, tool="mock")
    score_designs(ds, tool="mock")
    for d in ds:
        d.notes.append(strat)
    overlaps = [epitope_overlap(d.contact_residues, MAB_FOOTPRINT) for d in ds]
    summary.append(dict(strategy=strat, n=len(ds),
                        mean_mab_overlap=round(sum(overlaps)/len(overlaps), 3),
                        mean_pae=round(sum(d.pae_interaction for d in ds)/len(ds), 2),
                        mean_humanness=round(sum(d.humanness for d in ds)/len(ds), 3)))
epi_df = pd.DataFrame(summary)
print("Epitope-strategy comparison (SYNTHETIC mock metrics):")
epi_df

## 2 · Receptor-family specificity panel `[core]`

A TAA rarely lives alone: HER2 sits in the **HER/ErbB family** (EGFR/HER1, HER2, HER3, HER4) with
related surfaces. A useful VHH should prefer its target over the relatives. Model each survivor against
**each family member** and compare `pae_interaction`: a specific VHH has a clearly better (lower)
interface score for the target than for the off-targets. On mock this is SYNTHETIC plumbing; on Colab
run AF2-Multimer of each VHH vs each receptor (see `data/README.md` for the related-receptor panel).

In [ ]:
from antibody_tools import af2_multimer_ab

RECEPTOR_PANEL = ["HER2", "EGFR", "HER3", "HER4"]   # target + relatives (verify accessions in nb01)
TARGET = "HER2"

# Take the top few survivors from the overlapping campaign as the specificity test set.
test = design_vhh_cdrs(TARGET, EPITOPES["overlapping"], framework=FRAMEWORK, n=6, tool="mock")
score_designs(test, tool="mock")

spec_rows = []
for d in test:
    rec = {"design_id": d.design_id}
    for receptor in RECEPTOR_PANEL:
        m = af2_multimer_ab(d.sequence, antigen=receptor, tool="mock")
        rec[f"pae_{receptor}"] = m["pae_interaction"]
    # specificity margin: how much better the target is than the best off-target (lower pae = better)
    offtargets = [rec[f"pae_{r}"] for r in RECEPTOR_PANEL if r != TARGET]
    rec["spec_margin"] = round(min(offtargets) - rec[f"pae_{TARGET}"], 2)  # >0 => prefers target
    spec_rows.append(rec)
spec_df = pd.DataFrame(spec_rows)
print("Receptor-family specificity (SYNTHETIC mock pae_interaction; >0 spec_margin prefers target):")
spec_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(spec_df))
w = 0.2
for i, receptor in enumerate(RECEPTOR_PANEL):
    ax.bar(x + i*w, spec_df[f"pae_{receptor}"], width=w, label=receptor)
ax.set_xticks(x + 1.5*w); ax.set_xticklabels(spec_df["design_id"], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("pae_interaction (Å, mock)")
ax.set_title("Receptor-family specificity panel — SYNTHETIC (lower = better; want target lowest)")
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig("results/specificity_panel.png", dpi=150); plt.show()
print("Reminder: SYNTHETIC. On Colab, model each VHH vs each receptor with AF2-Multimer.")

## 3 · Developability + humanness figures `[core]`

Plot the survivors' **TAP-like** liability score, **CamSol-like** solubility, and **humanness** proxy.
These are **TEACHING HEURISTICS** (`antibody_tools.developability`), not the validated tools — the goal
is to teach the developability *axes* and to triage obvious liabilities (long CDR3, free cysteines,
N-glyc sequons, hydrophobic patches) **before** synthesis. Swap in real TAP/CamSol/Hu-mAb for any
reportable claim.

In [ ]:
camp = pd.read_csv("results/campaign.csv")

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].hist(camp["tap_score"], bins=12); ax[0].set_title("TAP-like liabilities (fewer better)")
ax[1].hist(camp["camsol_like"], bins=12); ax[1].set_title("CamSol-like solubility (higher better)")
ax[2].hist(camp["humanness"], bins=12); ax[2].set_title("humanness (higher better)")
plt.suptitle("Developability heuristics (NOT validated tools) — SYNTHETIC on mock")
plt.tight_layout(); plt.savefig("results/developability.png", dpi=150); plt.show()

# A simple developability triage flag (teaching): low liabilities AND humanized-enough.
camp["dev_ok"] = (camp["tap_score"] <= camp["tap_score"].median()) & (camp["humanness"] >= 0.5)
print("developability-OK (heuristic) fraction:", round(camp["dev_ok"].mean(), 3),
      "— TEACHING triage only; confirm with real TAP/CamSol/humanness tools.")

## D3 (part 2) checklist
- [ ] Epitope-choice comparison (overlapping vs non-overlapping) with what each strategy buys you.
- [ ] Receptor-family **specificity panel** (target vs relatives) + figure; specificity margin reported.
- [ ] Developability/humanness figures, flagged as **heuristics** (real tools for any claim).
- [ ] Every mock number labelled SYNTHETIC; conclusions phrased as plumbing/teaching, not results.

**Next:** `05_validation_plan.ipynb` — the display-screen plan + downstream format + controls.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — display screen, downstream format, controls

**Standard slot:** *validation plan.* **For Project 17 this means:** because de novo nanobody hit rates
are LOW, the deliverable is a **pooled display-screen plan** (designs → display → select on the TAA →
sequence winners → express → confirm), a **downstream construct** (VHH-Fc for imaging, or a CAR binder
module), a **receptor-family specificity panel**, and the mandatory **controls** (D4/D5).

This generates a structured plan file and a costed-reagent stub; it runs with no GPU.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Why a screen (not "we designed a binder")

A computational VHH design is a **hypothesis**. pLDDT is not affinity; a low pae_interaction is not
binding. De novo antibody/nanobody success rates are low, so the realistic pipeline is: **generate a
diverse pool → filter in silico → screen the pool experimentally (yeast/phage display) → recover and
characterize the winners.** Frame your designs as **screening inputs**.

## 1 · The display-screen plan `[core]`

A yeast-surface-display (or phage) screen of the filtered VHH pool against the labeled TAA. The plan
generator records the stages, the readout, and the controls so the plan is reproducible and gradable.

In [ ]:
import json, os

display_plan = {
    "format": "yeast surface display (Aga2p fusion) of the filtered VHH pool",
    "pool_source": "results/proj17_ranked.csv survivors (design_type='antibody')",
    "antigen_reagent": "recombinant TAA ectodomain (e.g., HER2 ECD), biotinylated, fluorophore-streptavidin",
    "stages": [
        "1. Synthesize the filtered VHH pool as an oligo library; clone into the display vector.",
        "2. Transform yeast; induce surface display; confirm display level (anti-tag stain).",
        "3. FACS round 1: select cells binding labeled TAA above a no-antigen gate.",
        "4. FACS rounds 2-3: increase stringency (lower antigen conc.) to enrich higher-affinity VHHs.",
        "5. Deep-sequence enriched pools; track per-design enrichment vs the input pool.",
        "6. Recover top VHHs; express solubly (see downstream format); confirm binding by SPR/BLI.",
    ],
    "readout": "FACS enrichment + NGS frequency; confirmatory SPR/BLI KD on recovered clones",
    "controls": {
        "positive": "a KNOWN anti-TAA nanobody (e.g., a published anti-HER2 VHH) spiked into the pool",
        "negative_irrelevant_antigen": "screen the same pool against an irrelevant antigen (e.g., BSA/"
                                       "a non-TAA protein) — winners must NOT enrich there",
        "negative_unrelated_binder": "an unrelated/non-binding VHH (display control, should not enrich)",
    },
    "specificity_gate": "counter-screen enriched VHHs against the receptor-family panel "
                        "(EGFR/HER3/HER4) — keep target-selective clones",
    "expectation": "LOW de novo hit rate; the screen is what turns a designed pool into real binders.",
}
os.makedirs("results", exist_ok=True)
with open("results/display_screen_plan.json", "w") as fh:
    json.dump(display_plan, fh, indent=2)
print("wrote results/display_screen_plan.json")
for s in display_plan["stages"]:
    print(" ", s)
print("\ncontrols:")
for k, v in display_plan["controls"].items():
    print(f"  {k}: {v}")

## 2 · Downstream format — imaging (VHH-Fc) or CAR binder `[core]`

A recovered, validated VHH is a *module*. Pick the translational format that matches your D0 goal:
- **VHH-Fc for imaging:** fuse the VHH to an Fc (avidity + longer half-life) or radiolabel the bare
  VHH for fast-clearing PET/SPECT tumor imaging.
- **CAR binder:** use the VHH as the antigen-binding domain of a CAR (the scFv-equivalent), linked to
  hinge/transmembrane + costimulatory + CD3ζ signaling domains.

This records the construct so the plan states what you would actually build.

In [ ]:
DOWNSTREAM = "VHH-Fc-imaging"   # or "CAR-binder"

constructs = {
    "VHH-Fc-imaging": {
        "construct": "VHH - (G4S)x linker - human IgG1 Fc (effector-silenced for imaging)",
        "purpose": "tumor-targeted imaging agent (PET/SPECT); Fc adds avidity + half-life",
        "format_notes": "bare VHH alternative for fast-clearing imaging; site-specific chelator for radiolabel",
        "assays": ["SPR/BLI KD on TAA", "cell binding on TAA+ vs TAA- lines", "(if lab) small-animal imaging"],
    },
    "CAR-binder": {
        "construct": "VHH - CD8 hinge/TM - 4-1BB - CD3zeta (VHH replaces the scFv as the binder)",
        "purpose": "CAR antigen-binding module against the TAA",
        "format_notes": "single-domain binder simplifies CAR design vs scFv; check tonic signaling",
        "assays": ["CAR-T cytotoxicity on TAA+ vs TAA- targets", "cytokine release", "specificity panel"],
    },
}
chosen = constructs[DOWNSTREAM]
print("downstream format:", DOWNSTREAM)
for k, v in chosen.items():
    print(f"  {k}: {v}")

## 3 · Controls + receptor-family specificity panel (mandatory) `[core]`

Controls are non-negotiable, even in the plan. State the **positive** (a known anti-TAA nanobody), the
**irrelevant-antigen negative** (winners must not enrich on a non-TAA protein), and the
**unrelated-binder negative** (display control). The **receptor-family specificity panel** (HER2 vs
EGFR/HER3/HER4) is the counter-screen that keeps target-selective clones.

In [ ]:
controls_and_specificity = {
    "controls": {
        "positive_control": "known anti-TAA VHH (e.g., a published anti-HER2 nanobody) — must enrich",
        "negative_irrelevant_antigen": "same pool vs BSA / a non-TAA protein — winners must NOT enrich",
        "negative_unrelated_binder": "non-binding/unrelated VHH on display — must NOT enrich",
    },
    "specificity_panel": ["HER2 (target)", "EGFR", "HER3", "HER4"],
    "specificity_decision": "keep VHHs that bind the target but not the relatives (target-selective)",
    "biparatopic_stretch": "pair a non-overlapping VHH with an overlapping one for a biparatopic/"
                           "bispecific construct (stronger avidity / dual-epitope engagement)  [stretch]",
}
import json
with open("results/controls_and_specificity.json", "w") as fh:
    json.dump(controls_and_specificity, fh, indent=2)
print(json.dumps(controls_and_specificity, indent=2))

## 4 · Costed reagent + timeline stub `[extension]`

A skeleton the student fills with real quotes/timelines. Numbers below are **EXAMPLE_DATA placeholders**
(not real quotes) — replace with vendor quotes in your D4 plan.

In [ ]:
import pandas as pd
# EXAMPLE_DATA placeholders — replace with real vendor quotes + your institution's timeline.
plan_items = pd.DataFrame([
    dict(item="VHH oligo library synthesis", purpose="display pool", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Display vector + yeast strain", purpose="surface display", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Recombinant TAA ECD (biotinylated)", purpose="FACS antigen", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="FACS sorting time", purpose="3 selection rounds", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="NGS of enriched pools", purpose="track enrichment", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Express + purify top clones", purpose="VHH-Fc / SPR", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="SPR/BLI KD on recovered clones", purpose="confirm binding", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
])
plan_items.to_csv("results/experimental_plan.csv", index=False)
print("wrote results/experimental_plan.csv (EXAMPLE_DATA placeholders — fill with real quotes)")
plan_items

## Responsible research (state in the plan)

This is a **therapeutic/diagnostic oncology** project — a designed nanobody against a human tumor
antigen for imaging or a CAR binder. In scope: diagnostic/therapeutic oncotargets. Out of scope:
anything enhancing pathogen transmissibility/virulence, toxins, or designs intended to cause harm. Any
real gene-synthesis order must go through a biosecurity-screening provider (IGSC member); wet-lab work
(including CAR-T) requires institutional biosafety/ethics approval. Do not overstate computational
designs as validated binders. See `MASTER_BLUEPRINT.md §7`.

## D4 / D5 checklist
- [ ] `results/display_screen_plan.json`: pooled yeast/phage display plan with stages + readout.
- [ ] Downstream construct chosen + recorded (VHH-Fc imaging **or** CAR binder).
- [ ] Controls (positive known nanobody, irrelevant-antigen negative, unrelated-binder negative) +
      receptor-family **specificity panel** specified.
- [ ] Costed reagent + timeline stub (EXAMPLE_DATA → real quotes).
- [ ] Responsible-research framing stated.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — and Projects 14–16 reuse this antibody-family pattern (RFantibody/BoltzGen → AF2-Multimer/
IgFold → developability → `design_type="antibody"` filter → display screen).